# Rhode Island 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Rhode Island, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note, there is no primary election dataset for Rhode Island 2008 so far.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total` , `dem_general_total`, `lib_general_total`, `grn_general_total`, `cst_general_total`, `psl_general_total`, `ind_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [2]:
# RI 2008 dataset path
# PRIMARY_PATH = r""
GENERAL_PATH = r"../../data/raw/2008/RI/20081104__ri__general__town.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/RI/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### b. General Election Dataset

In [20]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,town,office,district,party,candidate,votes
0,Bristol,Barrington,President,NaN,Democratic,Barack Obama,6075
1,Bristol,Barrington,President,NaN,Republican,John McCain,3666
2,Bristol,Barrington,President,NaN,Independent,Ralph Nader,78
3,Bristol,Barrington,President,NaN,Libertarian,Bob Barr,31
4,Bristol,Barrington,President,NaN,Green,Cynthia McKinney,15
5,Bristol,Barrington,President,NaN,Constitution,Chuck Baldwin,12
6,Bristol,Barrington,President,NaN,Socialism and Liberation,Glora La Riva,1
7,Bristol,Barrington,U.S. Senate,NaN,Democratic,John F. Reed,6627
8,Bristol,Barrington,U.S. Senate,NaN,Republican,Robert G. Tingle,2816
9,Bristol,Barrington,U.S. Representative,1.0,Democratic,Patrick J. Kennedy,5610


In [21]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President              273
State House            216
State Senate           127
U.S. Representative    100
U.S. Senate             78
State Question 1        78
State Question 2        76
State Question 3         2
Name: count, dtype: int64

In [22]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,town,office,district,party,candidate,votes
0,Bristol,Barrington,President,NaN,Democratic,Barack Obama,6075
1,Bristol,Barrington,President,NaN,Republican,John McCain,3666
2,Bristol,Barrington,President,NaN,Independent,Ralph Nader,78
3,Bristol,Barrington,President,NaN,Libertarian,Bob Barr,31
4,Bristol,Barrington,President,NaN,Green,Cynthia McKinney,15
5,Bristol,Barrington,President,NaN,Constitution,Chuck Baldwin,12
6,Bristol,Barrington,President,NaN,Socialism and Liberation,Glora La Riva,1
21,Bristol,Bristol,President,NaN,Democratic,Barack Obama,6833
22,Bristol,Bristol,President,NaN,Republican,John McCain,3834
23,Bristol,Bristol,President,NaN,Independent,Ralph Nader,112


In [23]:
# General data shape when only considering President
general_df.shape

(273, 7)

In [24]:
# Number of missing values in each column
general_df.isna().sum()

county         0
town           0
office         0
district     273
party          0
candidate      0
votes          0
dtype: int64

In [25]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,town,party,candidate,votes
0,Bristol,Barrington,Democratic,Barack Obama,6075
1,Bristol,Barrington,Republican,John McCain,3666
2,Bristol,Barrington,Independent,Ralph Nader,78
3,Bristol,Barrington,Libertarian,Bob Barr,31
4,Bristol,Barrington,Green,Cynthia McKinney,15
5,Bristol,Barrington,Constitution,Chuck Baldwin,12
6,Bristol,Barrington,Socialism and Liberation,Glora La Riva,1
7,Bristol,Bristol,Democratic,Barack Obama,6833
8,Bristol,Bristol,Republican,John McCain,3834
9,Bristol,Bristol,Independent,Ralph Nader,112


Since the RI records are town-level, we’ll group by county and sum the votes to obtain county totals.

In [26]:
# Groupby county vote counts
general_df = (
    general_df.groupby(["county", "candidate", "party"], as_index=False)["votes"].sum()
)

general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Bristol,Barack Obama,Democratic,16162
1,Bristol,Bob Barr,Libertarian,68
2,Bristol,Chuck Baldwin,Constitution,39
3,Bristol,Cynthia McKinney,Green,35
4,Bristol,Glora La Riva,Socialism and Liberation,9
5,Bristol,John McCain,Republican,9260
6,Bristol,Ralph Nader,Independent,234
7,Kent,Barack Obama,Democratic,48406
8,Kent,Bob Barr,Libertarian,242
9,Kent,Chuck Baldwin,Constitution,145


In [28]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Barack Obama        5
Bob Barr            5
Chuck Baldwin       5
Cynthia McKinney    5
Glora La Riva       5
John McCain         5
Ralph Nader         5
Name: count, dtype: int64

In [29]:
# Different parties in the general election data
general_df["party"].value_counts()

party
Democratic                  5
Libertarian                 5
Constitution                5
Green                       5
Socialism and Liberation    5
Republican                  5
Independent                 5
Name: count, dtype: int64

In [30]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [31]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Bristol,Barack Obama,Democratic,16162
1,Bristol,Bob Barr,Libertarian,68
2,Bristol,Chuck Baldwin,Constitution,39
3,Bristol,Cynthia McKinney,Green,35
4,Bristol,Glora La Riva,Socialism and Liberation,9
5,Bristol,John McCain,Republican,9260
6,Bristol,Ralph Nader,Independent,234
7,Kent,Barack Obama,Democratic,48406
8,Kent,Bob Barr,Libertarian,242
9,Kent,Chuck Baldwin,Constitution,145


In [32]:
# Shape after preprocessing
general_df.shape

(35, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [38]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democratic"               : "dem",                     
                "Republican"               : "rep",
                "Libertarian"              : "lib",
                "Green"                    : "grn",
                "Constitution"             : "cst",
                "Socialism and liberation" : "psl",
                "Independent"              : "ind"
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [39]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [40]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [43]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_psl_RIVA,gen_rep_MCCAIN
0,Bristol,39,16162,35,234,68,9,9260
1,Kent,145,48406,139,943,242,23,32780
2,Newport,66,25479,78,366,127,9,15717
3,Providence,336,167442,428,2592,677,68,81010
4,Washington,89,39082,117,694,268,13,25624


In [44]:
# General dataframe shape after pivot
general_pivot.shape

(5, 8)

## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns
* `lib_general_total` = sum of all `gen_lib_*` columns
* `grn_general_total` = sum of all `gen_grn_*` columns
* `cst_general_total` = sum of all `gen_cst_*` columns
* `psl_general_total` = sum of all `gen_psl_*` columns
* `ind_general_total` = sum of all `gen_ind_*` columns

In [46]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
grn_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_grn")]
cst_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_cst")]
psl_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_psl")]
ind_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_ind")]

general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["grn_general_total"] = general_pivot[grn_general_cols].sum(axis=1) if grn_general_cols else 0
general_pivot["cst_general_total"] = general_pivot[cst_general_cols].sum(axis=1) if cst_general_cols else 0
general_pivot["psl_general_total"] = general_pivot[psl_general_cols].sum(axis=1) if psl_general_cols else 0
general_pivot["ind_general_total"] = general_pivot[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [47]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned general dataframe:")
general_pivot.columns

Final columns in the cleaned general dataframe:


Index(['county', 'gen_cst_BALDWIN', 'gen_dem_OBAMA', 'gen_grn_MCKINNEY',
       'gen_ind_NADER', 'gen_lib_BARR', 'gen_psl_RIVA', 'gen_rep_MCCAIN',
       'rep_general_total', 'dem_general_total', 'lib_general_total',
       'grn_general_total', 'cst_general_total', 'psl_general_total',
       'ind_general_total'],
      dtype='object')

In [48]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_psl_RIVA,gen_rep_MCCAIN,rep_general_total,dem_general_total,lib_general_total,grn_general_total,cst_general_total,psl_general_total,ind_general_total
0,Bristol,39,16162,35,234,68,9,9260,9260,16162,68,35,39,9,234
1,Kent,145,48406,139,943,242,23,32780,32780,48406,242,139,145,23,943
2,Newport,66,25479,78,366,127,9,15717,15717,25479,127,78,66,9,366
3,Providence,336,167442,428,2592,677,68,81010,81010,167442,677,428,336,68,2592
4,Washington,89,39082,117,694,268,13,25624,25624,39082,268,117,89,13,694


Now, we save the cleaned dataframe into the processed directory.

In [49]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "RI.csv", index=False)